In [ ]:
!pip install -U transformers datasets accelerate sentencepiece evaluate --quiet

# imports

In [ ]:
import os
import gc
import random
import shutil
import time
from dataclasses import dataclass
from typing import Dict, List, Optional
import evaluate
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR
from torch.optim.optimizer import Optimizer
import math
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorForSeq2Seq,
)

try:
    from safetensors.torch import load_file as safe_load, save_file as safe_save
except Exception:
    safe_load, safe_save = None, None

from transformers.utils import logging
logging.set_verbosity_error()   # hides WARNING/INFO messages (incl. load reports)


In [ ]:
def cosine_lr(initial_lr: float,
              round_idx: int,
              total_rounds: int,
              eta_min: float = 0.0) -> float:
    round_idx = min(round_idx, total_rounds)
    cosine_decay = 0.5 * (1 + math.cos(math.pi * round_idx / total_rounds))
    return eta_min + (initial_lr - eta_min) * cosine_decay

# Constants

In [ ]:
seed = 342365   # paper seeds: 342365, 8541865, 785638984, 1486496, 2658  (one per run)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



MODEL_NAME = "google/electra-small-discriminator"
Main_path = "./main_model"


OUTPUT_DIR = "electra"
MAIN_PATH_MODEL = Main_path
global_adapter_dir = Main_path
ADAPTER_BIN = "adapter_model.bin"
ADAPTER_SAFE = "adapter_model.safetensors"

EPOCHS = 1
LR = 1.1e-4
rounds = 100
NUM_LABELS = 2
MAX_LENGTH = 96
BATCH_SIZE = 512
num_of_client = 100
num_of_Part = 10

# Kaggle 'GPU T4 x2': full fine-tuning, plain fp32 (no AMP / no GradScaler), DataParallel over both GPUs
N_GPU = torch.cuda.device_count()
print('GPUs visible:', N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])


# sophia

In [ ]:
class SophiaG(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.965, 0.99), rho = 0.04,
         weight_decay=1e-1, *, maximize: bool = False,
         capturable: bool = False):
        if not 0.0 <= lr:
            raise ValueError("Invalid learning rate: {}".format(lr))
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError("Invalid beta parameter at index 0: {}".format(betas[0]))
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError("Invalid beta parameter at index 1: {}".format(betas[1]))
        if not 0.0 <= rho:
            raise ValueError("Invalid rho parameter at index 1: {}".format(rho))
        if not 0.0 <= weight_decay:
            raise ValueError("Invalid weight_decay value: {}".format(weight_decay))
        defaults = dict(lr=lr, betas=betas, rho=rho,
                        weight_decay=weight_decay,
                        maximize=maximize, capturable=capturable)
        super(SophiaG, self).__init__(params, defaults)

    def __setstate__(self, state):
        super().__setstate__(state)
        for group in self.param_groups:
            group.setdefault('maximize', False)
            group.setdefault('capturable', False)
        state_values = list(self.state.values())
        step_is_tensor = (len(state_values) != 0) and torch.is_tensor(state_values[0]['step'])
        if not step_is_tensor:
            for s in state_values:
                s['step'] = torch.tensor(float(s['step']))

    @torch.no_grad()
    def update_hessian(self):
        for group in self.param_groups:
            beta1, beta2 = group['betas']
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]

                if len(state) == 0:
                    state['step'] = torch.zeros((1,), dtype=torch.float, device=p.device) \
                        if self.defaults['capturable'] else torch.tensor(0.)
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state['hessian'] = torch.zeros_like(p, memory_format=torch.preserve_format)

                if 'hessian' not in state.keys():
                    state['hessian'] = torch.zeros_like(p, memory_format=torch.preserve_format)

                state['hessian'].mul_(beta2).addcmul_(p.grad, p.grad, value=1 - beta2)


    @torch.no_grad()
    def step(self, closure=None, bs=5120):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            params_with_grad = []
            grads = []
            exp_avgs = []
            state_steps = []
            hessian = []
            beta1, beta2 = group['betas']

            for p in group['params']:
                if p.grad is None:
                    continue
                params_with_grad.append(p)

                if p.grad.is_sparse:
                    raise RuntimeError('Hero does not support sparse gradients')
                grads.append(p.grad)
                state = self.state[p]
                # State initialization
                if len(state) == 0:
                    state['step'] = torch.zeros((1,), dtype=torch.float, device=p.device) \
                        if self.defaults['capturable'] else torch.tensor(0.)
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state['hessian'] = torch.zeros_like(p, memory_format=torch.preserve_format)

                if 'hessian' not in state.keys():
                    state['hessian'] = torch.zeros_like(p, memory_format=torch.preserve_format)

                exp_avgs.append(state['exp_avg'])
                state_steps.append(state['step'])
                hessian.append(state['hessian'])

                if self.defaults['capturable']:
                    bs = torch.ones((1,), dtype=torch.float, device=p.device) * bs

            sophiag(params_with_grad,
                  grads,
                  exp_avgs,
                  hessian,
                  state_steps,
                  bs=bs,
                  beta1=beta1,
                  beta2=beta2,
                  rho=group['rho'],
                  lr=group['lr'],
                  weight_decay=group['weight_decay'],
                  maximize=group['maximize'],
                  capturable=group['capturable'])

        return loss

def sophiag(params: List[Tensor],
          grads: List[Tensor],
          exp_avgs: List[Tensor],
          hessian: List[Tensor],
          state_steps: List[Tensor],
          capturable: bool = False,
          *,
          bs: int,
          beta1: float,
          beta2: float,
          rho: float,
          lr: float,
          weight_decay: float,
          maximize: bool):

    if not all(isinstance(t, torch.Tensor) for t in state_steps):
        raise RuntimeError("API has changed, `state_steps` argument must contain a list of singleton tensors")


    func = _single_tensor_sophiag

    func(params,
         grads,
         exp_avgs,
         hessian,
         state_steps,
         bs=bs,
         beta1=beta1,
         beta2=beta2,
         rho=rho,
         lr=lr,
         weight_decay=weight_decay,
         maximize=maximize,
         capturable=capturable)

def _single_tensor_sophiag(params: List[Tensor],
                         grads: List[Tensor],
                         exp_avgs: List[Tensor],
                         hessian: List[Tensor],
                         state_steps: List[Tensor],
                         *,
                         bs: int,
                         beta1: float,
                         beta2: float,
                         rho: float,
                         lr: float,
                         weight_decay: float,
                         maximize: bool,
                         capturable: bool):

    for i, param in enumerate(params):
        grad = grads[i] if not maximize else -grads[i]
        exp_avg = exp_avgs[i]
        hess = hessian[i]
        step_t = state_steps[i]

        if capturable:
            assert param.is_cuda and step_t.is_cuda and bs.is_cuda

        if torch.is_complex(param):
            grad = torch.view_as_real(grad)
            exp_avg = torch.view_as_real(exp_avg)
            hess = torch.view_as_real(hess)
            param = torch.view_as_real(param)

        # update step
        step_t += 1

        # Perform stepweight decay
        param.mul_(1 - lr * weight_decay)

        # Decay the first and second moment running average coefficient
        exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)

        if capturable:
            step_size = lr
            step_size_neg = step_size.neg()

            ratio = (exp_avg.abs() / (rho * bs * hess + 1e-15)).clamp(None,1)
            param.addcmul_(exp_avg.sign(), ratio, value=step_size_neg)
        else:
            step_size_neg = - lr

            ratio = (exp_avg.abs() / (rho * bs * hess + 1e-15)).clamp(None,1)
            param.addcmul_(exp_avg.sign(), ratio, value=step_size_neg)

# model Import

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=NUM_LABELS)
model.to(device)
model.save_pretrained(MAIN_PATH_MODEL)
model.save_pretrained("./pre_model")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(trainable_params/total_params)

# Data

In [ ]:
import pandas as pd

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

In [ ]:
df_test = pd.read_parquet('/kaggle/input/datasets/aaserfawzy/sentiment-orme/0000.parquet')
df_train = pd.read_parquet('/kaggle/input/datasets/aaserfawzy/sentiment-orme/0000.parquet.1')
df_train.info()
df_test  = df_test[df_test["sentiment"] != 2]
df_train["label"] = (df_train["sentiment"] == 4).astype("int8")
df_test["label"]  = (df_test["sentiment"] == 4).astype("int8")

X = df_train.drop(columns=["label"])
y = df_train["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.005,
    stratify=y,
    random_state=seed
)

train_df = X_train.copy()
train_df["label"] = y_train

test_df = X_test.copy()
test_df["label"] = y_test
print(len(train_df),len(test_df))

In [ ]:
label_counts = df_test["label"].astype(int).value_counts().sort_index()

print(f"Number of data points {len(df_test)}")
print("Number of data points per label in test_dataset:")

for label, count in label_counts.items():
    print(f"Label {label}: {count} data points")

In [ ]:
label_counts = df_train["label"].astype(int).value_counts().sort_index()

print(f"Number of data points {len(df_train)}")
print("Number of data points per label in test_dataset:")

for label, count in label_counts.items():
    print(f"Label {label}: {count} data points")

In [ ]:
label_counts = test_df["label"].astype(int).value_counts().sort_index()

print(f"Number of data points {len(test_df)}")
print("Number of data points per label in test_dataset:")

for label, count in label_counts.items():
    print(f"Label {label}: {count} data points")

In [ ]:
hf_valid = Dataset.from_pandas(test_df, preserve_index=False)
hf_test  = Dataset.from_pandas(df_test, preserve_index=False)

def preprocess_fn(examples):
    encodings = tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=MAX_LENGTH,
    )
    encodings["labels"] = examples["label"]
    return encodings

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
cols = ["input_ids", "attention_mask", "labels"]

tokenized_valid = hf_valid.map(preprocess_fn, batched=True, remove_columns=hf_valid.column_names)
tokenized_test  = hf_test.map(preprocess_fn, batched=True, remove_columns=hf_test.column_names)

tokenized_valid.set_format("torch", columns=cols)
tokenized_test.set_format("torch", columns=cols)

valid_loader = DataLoader(tokenized_valid, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test_loader  = DataLoader(tokenized_test,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)


# functions

In [ ]:
s = np.array([np.random.choice(np.arange(num_of_client), size=10, replace=False) for _ in range(200)])
def get_trained_client(setting,numberOfClient,numberParticpate):

    clients = list(range(numberOfClient))

    if setting == 1:
        return clients

    elif setting == 2:
        return  random.sample(clients, numberParticpate)

    elif setting == 3:
        count = random.randint(1, len(clients))
        return random.sample(clients, count)
    elif setting == 4:
        senario = s
        return senario[numberParticpate]
    else:
        raise ValueError("no existing setting that can be used")

In [ ]:
@torch.no_grad()
def evaluate_valid_loss(model, loader: DataLoader, device):
    """
    Computes mean token-level negative log-likelihood (NLL)
    over the validation set, without using HuggingFace's
    built-in loss.

    Returns:
        valid_loss (float): mean NLL over all non-masked tokens
    """
    model.eval()

    total_nll = 0.0
    total_tokens = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        # logits: [batch_size, seq_len, vocab_size]
        logits = outputs.logits

        # Compute log-probabilities
        log_probs = F.log_softmax(logits, dim=-1)

        # Flatten tensors
        labels_flat = labels.view(-1)
        log_probs_flat = log_probs.view(-1, log_probs.size(-1))

        # Mask out ignored labels (-100)
        valid_mask = labels_flat != -100
        valid_labels = labels_flat[valid_mask]

        # Gather log-probabilities of the true labels
        token_log_probs = log_probs_flat[valid_mask, valid_labels]

        # Accumulate NLL
        total_nll += -token_log_probs.sum()
        total_tokens += valid_labels.numel()

    # Mean NLL over entire validation set
    valid_loss = (total_nll / total_tokens).item()

    return valid_loss

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def evaluate_model(model, device, test_loader):
    """
    Hugging Face–compatible evaluation function

    Args:
        model: Hugging Face model (e.g. AutoModelForSequenceClassification)
        device: torch.device
        test_loader: DataLoader (HF-style batches)
    Returns:
        dict with validation_loss, accuracy, f1
    """
    model.to(device)
    model.eval()

    all_logits = []
    all_labels = []
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for batch in test_loader:
            # HF batches are dicts
            inputs = {k: v.to(device) for k, v in batch.items()}

            # Forward pass (HF computes loss internally)
            outputs = model(**inputs)

            loss = outputs.loss
            logits = outputs.logits
            labels = inputs["labels"]

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_logits.append(logits.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    avg_loss = total_loss / total_samples

    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    preds = np.argmax(all_logits, axis=1)

    metrics = {
        "validation_loss": avg_loss,
        "accuracy": accuracy.compute(
            predictions=preds,
            references=all_labels
        )["accuracy"],
        "f1": f1.compute(
            predictions=preds,
            references=all_labels
        )["f1"]
    }

    return metrics

In [ ]:
def _int_counts_from_simplex(prob, total, rng):
    """
    Convert simplex vector prob into integer counts that sum to total
    using floor + remainder distribution.
    """
    raw = prob * total
    counts = np.floor(raw).astype(int)
    rem = total - counts.sum()
    if rem > 0:
        frac = raw - counts
        # deterministic tie-breaking with rng noise
        order = np.argsort(-frac + 1e-12 * rng.random(len(frac)))
        counts[order[:rem]] += 1
    return counts

def dirichlet_partition_both(
    labels,
    num_clients: int = 10,
    alpha: float = 0.5,
    beta: float | None = None,
    prior: str = "uniform",      # you asked for uniform
    min_client_size: int = 1,
    seed: int = 42,
):

    rng = np.random.default_rng(seed)
    y = np.asarray(labels)
    N = len(y)

    # Map labels to 0..L-1
    classes = np.unique(y)
    L = len(classes)
    to_int = {c: i for i, c in enumerate(classes)}
    y_int = np.array([to_int[v] for v in y], dtype=int)

    # Prior p over labels (uniform per your requirement)
    if prior == "uniform":
        p = np.ones(L) / L
    elif prior == "empirical":
        cnt = np.bincount(y_int, minlength=L)
        p = cnt / cnt.sum()
    else:
        raise ValueError("prior must be 'uniform' or 'empirical'.")

    # ----- quantity sizes -----
    if beta is None:
        # balanced: same or very similar number of examples
        base = N // num_clients
        rem = N % num_clients
        sizes = [base + (1 if i < rem else 0) for i in range(num_clients)]
        z = None
    else:

        z = rng.dirichlet(np.ones(num_clients) * beta)
        sizes = _int_counts_from_simplex(z, N, rng).tolist()

        # enforce min_client_size if requested
        if min_client_size > 1:
            if min_client_size * num_clients > N:
                raise ValueError("min_client_size too large for total dataset size.")
            # raise small clients up to min by taking from larger ones
            deficit = sum(max(0, min_client_size - s) for s in sizes)
            if deficit > 0:
                order = np.argsort(-np.array(sizes))
                for i in order:
                    if deficit <= 0:
                        break
                    spare = sizes[i] - min_client_size
                    if spare > 0:
                        take = min(spare, deficit)
                        sizes[i] -= take
                        deficit -= take
                for j in range(num_clients):
                    need = max(0, min_client_size - sizes[j])
                    sizes[j] += need

            # re-fix sum exactly to N
            diff = N - sum(sizes)
            if diff > 0:
                for _ in range(diff):
                    sizes[rng.integers(num_clients)] += 1
            elif diff < 0:
                diff = -diff
                for _ in range(diff):
                    candidates = [i for i, s in enumerate(sizes) if s > min_client_size]
                    i = rng.choice(candidates)
                    sizes[i] -= 1

    # ----- label distributions -----

    conc = np.clip(alpha * p, 1e-12, None)  # ensure positive concentration
    q = rng.dirichlet(conc, size=num_clients)

    # Pools of indices per label (for sampling without replacement)
    pools = []
    for k in range(L):
        idxs = np.where(y_int == k)[0]
        rng.shuffle(idxs)
        pools.append(list(idxs))

    # Allocate without replacement according to q_j
    client_indices = []
    for j in range(num_clients):
        m = sizes[j]
        desired = rng.multinomial(m, q[j])

        assigned = []
        # take desired from each pool
        for k in range(L):
            take = min(desired[k], len(pools[k]))
            if take > 0:
                assigned.extend(pools[k][:take])
                pools[k] = pools[k][take:]

        while len(assigned) < m:
            avail = np.array([len(pools[k]) for k in range(L)])
            if avail.sum() == 0:
                break
            mask = (avail > 0).astype(float)
            probs = q[j] * mask
            probs = probs / probs.sum()
            k = rng.choice(L, p=probs)
            assigned.append(pools[k].pop())  # remove => without replacement

        rng.shuffle(assigned)
        client_indices.append(assigned)

    return client_indices, q, sizes, classes, z

In [ ]:

def make_client_shards(df, client_indices):
    """
    df: pandas.DataFrame
    client_indices: list[list[int]]  (row indices per client)
    returns: list[pandas.DataFrame]
    """
    return [df.iloc[idxs].reset_index(drop=True) for idxs in client_indices]


In [ ]:
client_idxs, q, sizes, classes, z = dirichlet_partition_both(
    labels=train_df["label"],
    num_clients=num_of_client,
    alpha=0.5,          # strong label non-IID
    beta=None,          # balanced client sizes
    prior="uniform",
    seed=seed
)
print("sizes:", sizes, "min/max:", min(sizes), max(sizes))

In [ ]:
all_ids = np.concatenate([np.array(x) for x in client_idxs])
print("Assigned:", len(all_ids), "Unique:", len(np.unique(all_ids)))

In [ ]:
client_shards = make_client_shards(train_df, client_idxs)

print("Client sizes:", [len(c) for c in client_shards])
print("Min/Max:", min(len(c) for c in client_shards), max(len(c) for c in client_shards))

print("Example client[0]:")
print(client_shards[0].iloc[0])


In [ ]:
train_proc = [ Dataset.from_pandas(train_ds, preserve_index=False) for train_ds in client_shards]
train_proc = [ train_ds.map(preprocess_fn, batched=True, remove_columns=train_ds.column_names) for train_ds in train_proc]
for i in train_proc:
        i.set_format(type="torch", columns=cols)

In [ ]:
train_loader = [DataLoader(i, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator) for i in train_proc]

# model realated function

In [ ]:
def base_model_loader(adapter_dir: str):
    model = AutoModelForSequenceClassification.from_pretrained(adapter_dir,num_labels=2)
    return model

In [ ]:
def _move_optimizer_state_to_device(opt_state, device):
    for state in opt_state.get("state", {}).values():
        for k, v in list(state.items()):
            if torch.is_tensor(v):
                state[k] = v.to(device, non_blocking=True)
            elif isinstance(v, (list, tuple)):
                state[k] = [t.to(device, non_blocking=True) if torch.is_tensor(t) else t for t in v]
            elif isinstance(v, dict):
                for kk, vv in list(v.items()):
                    if torch.is_tensor(vv):
                        v[kk] = vv.to(device, non_blocking=True)

In [ ]:
def save_full_model_state_dict(state_dict, out_dir: str):
    """
    Save FULL model weights (HF-compatible).
    Deletes existing directory to avoid stale files.
    """
    if os.path.exists(out_dir):
        print("removed")
        shutil.rmtree(out_dir)

    os.makedirs(out_dir, exist_ok=True)

    out_path = os.path.join(out_dir, "pytorch_model.bin")
    torch.save(state_dict, out_path)


def load_full_model_state_dict(model_dir: str, map_location="cpu"):
    """
    Load FULL model weights from HF-style directory.
    """
    bin_path = os.path.join(model_dir, "pytorch_model.bin")
    safe_path = os.path.join(model_dir, "model.safetensors")

    if os.path.exists(bin_path):
        return torch.load(bin_path, map_location=map_location)
    elif safe_load is not None and os.path.exists(safe_path):
        return safe_load(safe_path)
    else:
        raise FileNotFoundError(f"No model weights found in {model_dir}")


# ============================================================
# 2) FEDAVG OVER FULL MODELS (CPU SAFE)
# ============================================================

@torch.no_grad()
def average_full_model_states(model_dirs, weights=None):
    """
    FedAvg over FULL model parameters on CPU.
    Safely skips non-floating tensors.
    """
    n = len(model_dirs)
    if n == 0:
        raise ValueError("No model dirs provided.")

    if weights is None:
        weights = [1.0 / n] * n
    else:
        s = sum(weights)
        weights = [w / s for w in weights]

    avg = load_full_model_state_dict(model_dirs[0], map_location="cpu")

    for k, v in avg.items():
        if torch.is_floating_point(v):
            avg[k] = v.float() * weights[0]

    for w, d in zip(weights[1:], model_dirs[1:]):
        sd = load_full_model_state_dict(d, map_location="cpu")
        for k in avg:
            if torch.is_floating_point(avg[k]):
                avg[k].add_(sd[k].float(), alpha=w)

    return avg


# ============================================================
# 4) AGGREGATION CONTROLLER (FULL MODELS ONLY)
# ============================================================
def add_parameters(model, edge, ratio):
  for server_param, edge_param in zip(model.parameters(), edge.parameters()):
    server_param.data += edge_param.data.clone() * ratio

def aggregate_adapters(
    selected_model_dirs,
    algorithm: str,
    global_model_dir: str,
    train_loaders=None,
    client_ids=None,
    wagg_set=None,
    base_model_loader=None,
    device="cuda",
):
    """
    Supported algorithms (FULL MODEL):
      - Sophia   (uniform FedAvg)
      - FedAvg   (data-size weighted)
      - wagg     (NLL-weighted)
    """
    model = base_model_loader(global_model_dir)
    # ============= SOPHIA =================
    if algorithm.lower() == "sophia":
      for p in model.parameters():
          p.data.zero_()
      # equal‐weight average over selected edges
      for edge in selected_model_dirs:
          cloneModel = base_model_loader(edge)
          add_parameters(model,cloneModel, 1 / len(selected_model_dirs))
          del cloneModel                # Remove the variable reference
          torch.cuda.empty_cache()

      model.save_pretrained(global_model_dir)
      del model                # Remove the variable reference
      torch.cuda.empty_cache()

      return

    # ============= FEDAVG =================
    if algorithm.lower() == "fedavg":
      for p in model.parameters():
          p.data.zero_()
      # equal‐weight average over selected edges
      for i,edge in enumerate(selected_model_dirs):
          cloneModel = base_model_loader(edge)
          add_parameters(model,cloneModel, 1 / len(train_loaders[i]))
          del cloneModel                # Remove the variable reference
          torch.cuda.empty_cache()

      model.save_pretrained(global_model_dir)
      del model                # Remove the variable reference
      torch.cuda.empty_cache()
      return

    # ============= WAGG ===================
    if algorithm.lower() == "wagg":

      assert wagg_set is not None
      assert base_model_loader is not None

      selected_model_dirs.append(global_model_dir)

      for p in model.parameters():
        p.data.zero_()

      scores = []
      weights = []
      for d in selected_model_dirs:
        cloneModel = base_model_loader(d).to(device)
        evalModel = torch.nn.DataParallel(cloneModel) if N_GPU > 1 else cloneModel
        loss = evaluate_valid_loss(evalModel, wagg_set, device)
        scores.append(1.0 / loss)
        del evalModel

        del cloneModel
        torch.cuda.empty_cache()
        gc.collect()

        s = sum(scores)
        weights = [x / s for x in scores] if s > 0 else [1.0 / len(scores)] * len(scores)

      for i,edge in enumerate(selected_model_dirs):
        cloneModel = base_model_loader(edge)
        add_parameters(model,cloneModel,weights[i])
        del cloneModel                # Remove the variable reference
        torch.cuda.empty_cache()

      model.save_pretrained(global_model_dir)
      del model                # Remove the variable reference
      torch.cuda.empty_cache()
      return

    # ============= FEDLAW ==================
    if algorithm.lower() == "fedlaw":
        raise RuntimeError(
            "FedLAW is adapter-only. "
            "Scaling and fusing FULL model parameters is unstable "
            "(LayerNorm / embeddings break)."
        )

    raise ValueError(f"Unknown algorithm: {algorithm}")

In [ ]:
def get_client_state_paths(edge: int, base_state_dir="client_state"):
    state_dir = os.path.join(base_state_dir, f"edge_{edge}")
    os.makedirs(state_dir, exist_ok=True)
    opt_path = os.path.join(state_dir, "optimizer.pt")
    return state_dir, opt_path

def get_round_adapter_dir(round_id: int, edge: int, cache_root="round_cache"):
    d = os.path.join(cache_root, f"round_{round_id}", f"edge_{edge}")
    os.makedirs(d, exist_ok=True)
    return d

# train loop

In [ ]:
print("main")
test_acc = evaluate_model(model, device,test_loader)
print(f" global_loss={test_acc["validation_loss"]:.4f} global_acc = {test_acc["accuracy"]:.4f} f1={test_acc["f1"]:.4f}")


In [ ]:
def train_secondorder_fedsophia(
    base_model_loader,          # function(adapter_dir) -> model (base + adapter loaded)
    global_adapter_dir: str,    # current global adapter checkpoint directory
    train_loaders,
    test_loader,
    valid_loaders=None,         # if you use it elsewhere
    algorithm="Sophia",
    grad_clip=1.0,
    lr=3e-5,
    epochs=1,
    rounds=10,
    num_clients=10,
    clients_per_round=4,
    seed=42,
    device="cuda",
    k = 10,
    round0_clients=None,   # override S_0 (used for the probe / warm-up round)
    reuse_round0=False,    # Alg.1 line 3: w_{0,i} <- w~_i, take the round-0 client models from the probe stage
    probe_only=False,      # Alg.1 line 1: stop after round-0 local training (no aggregation)
    stop_after=None):      # run only the first N rounds but keep the lr schedule of `rounds` (used by the K sweep)

    results = []

    for globalRound in range(stop_after or rounds):
        print(f"\n##################  Round {globalRound}  ########################")

        chosen = get_trained_client(4, num_clients, globalRound)
        if globalRound == 0 and round0_clients is not None:
            chosen = round0_clients

        selected_adapter_dirs = []
        selected_client_ids = []

        for edge in range(num_clients):
            if edge not in chosen:
                continue

            print(f"\n################## edge {edge} ########################")

            # ephemeral adapter output for this round (can delete later)
            out_adapter_dir = get_round_adapter_dir(globalRound, edge)
            _ , opt_path = get_client_state_paths(edge)
            if globalRound == 0 and reuse_round0:
                assert os.path.exists(os.path.join(out_adapter_dir, "config.json")), f"round-0 model of edge {edge} missing"
                print("reusing round-0 model from the probe stage")
                selected_adapter_dirs.append(out_adapter_dir)
                selected_client_ids.append(edge)
                continue
            # ---- load model (base + current global adapter) to GPU ----
            model = base_model_loader(adapter_dir=global_adapter_dir).to(device)
            model.train()
            net = torch.nn.DataParallel(model) if N_GPU > 1 else model   # splits each batch over both GPUs
            new_lr = cosine_lr(initial_lr=lr,round_idx=globalRound,total_rounds=rounds,eta_min=5e-8)
            params = [p for p in model.parameters() if p.requires_grad]
            #print(model.print_trainable_parameters())
            optimizer = SophiaG( params, lr=new_lr, weight_decay=5e-7)
            if os.path.exists(opt_path):
                opt_state = torch.load(opt_path, map_location="cpu")
                _move_optimizer_state_to_device(opt_state, device)
                optimizer.load_state_dict(opt_state)
                print(f"Loaded optimizer state for edge {edge} from {opt_path}")

            for param_group in optimizer.param_groups:
                param_group["lr"] = new_lr

            global_steps = 0
            # ---- local training ----
            for epoch in range(1, epochs + 1):
                running_loss = 0.0

                for step, batch in enumerate(train_loaders[edge], start=1):
                    input_ids = batch["input_ids"].to(device, non_blocking=True)
                    attention_mask = batch["attention_mask"].to(device, non_blocking=True)
                    labels = batch["labels"].to(device, non_blocking=True)

                    optimizer.zero_grad(set_to_none=True)

                    outputs = net(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels
                    )
                    loss = outputs.loss.mean()   # DataParallel returns one loss per GPU
                    loss.backward()

                    if grad_clip and grad_clip > 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

                    optimizer.step()
                    running_loss += float(loss.detach().cpu())
                    global_steps = global_steps + 1
                    # ---- Hessian update (cheap-ish) ----
                    # IMPORTANT: no graph should be kept alive from above.
                    if global_steps % k != 0 :
                        continue
                    else:

                        logits = net(input_ids=input_ids, attention_mask=attention_mask,labels=labels).logits
                        # sample pseudo-labels from model distribution
                        y_sample = torch.distributions.Categorical(logits=logits).sample()

                        loss_sampled = F.cross_entropy(logits.view(-1, logits.size(-1)), y_sample.view(-1), ignore_index=-1)
                        loss_sampled.backward()

                        # Update Hessian estimate in Sophia
                        optimizer.update_hessian()
                        optimizer.zero_grad(set_to_none=True)
                        model.zero_grad()


                avg_train_loss = running_loss / max(1, len(train_loaders[edge]))
                val_acc= evaluate_model(model, device, test_loader)  # you can swap to a smaller client-val loader
                print(f"Epoch {epoch}: train_loss={avg_train_loss:.4f}  val_loss={val_acc["validation_loss"]:.4f} acc={val_acc["accuracy"]:.4f} f1={val_acc["f1"]:.4f}")

            # ---- save ONLY adapter + optimizer state ----
            model.save_pretrained(out_adapter_dir)
            torch.save(optimizer.state_dict(), opt_path)

            selected_adapter_dirs.append(out_adapter_dir)
            selected_client_ids.append(edge)

            # ---- free GPU memory (your requirement) ----
            del net
            del model
            del optimizer
            torch.cuda.empty_cache()
            gc.collect()

        if probe_only:
            return selected_adapter_dirs

        # ---- server aggregation (adapter-only, CPU) ----
        aggregate_adapters(
            selected_adapter_dirs,
            algorithm=algorithm,
            global_model_dir=global_adapter_dir,  # overwrite/update
            train_loaders=train_loaders,
            client_ids=selected_client_ids,
            wagg_set=valid_loaders if valid_loaders is not None else None,
            base_model_loader=base_model_loader,
            device=device,
        )

        # ---- evaluate global model ----
        global_model = base_model_loader(adapter_dir=global_adapter_dir).to(device)
        test_acc = evaluate_model(global_model, device,test_loader)
        #global_model.save_pretrained(global_adapter_dir)
        results.append(test_acc)
        print(f"Round {globalRound}: global_loss={test_acc["validation_loss"]:.4f} global_acc = {test_acc["accuracy"]:.4f} f1={test_acc["f1"]:.4f}")

        del global_model
        torch.cuda.empty_cache()
        gc.collect()

        # ---- delete per-round cache to save disk (optional) ----
        round_dir = os.path.join("round_cache", f"round_{globalRound}")
        if os.path.exists(round_dir) and not(globalRound == rounds-1):
            shutil.rmtree(round_dir, ignore_errors=True)

    return results

# FedVARE ablation: reference-set selection (Sentiment140, non-IID, Sophia, full fine-tuning)

Follows Algorithm 1 of the paper:

1. **Line 1** - round 0: the clients in $S_0$ train locally from $w^0$ (once, shared by every variant).
2. **Line 2** - the server scores every $z \in D_{eval}$ with $V(z)=\mathrm{Var}_i[\ell(\tilde w_i; z)]$ and builds the reference sets.
3. **Line 3** - each variant starts from **the same** round-0 client models $w_{0,i}=\tilde w_i$, so the only thing that differs between runs is the reference set.
4. **Lines 4-11** - 100 rounds of inverse-loss aggregation (global model included) on that variant's reference set.

| variant | reference set ($D^*_{val}$) |
|---|---|
| `high_variance` | top-50 $V(z)$ per label (100) - FedVARE |
| `mid_variance` | 50 per label closest to the label mean $\mu_c$ (100) |
| `low_variance` | 50 per label with the lowest $V(z)$ (100) |
| `random` | 50 per label uniformly at random (100) |
| `full` | all of $D_{eval}$ (no selection) |

# VarNLL scoring and selection (from the selection notebook)

In [ ]:
# ================================================================
# VarNLL Selection Pipeline (ONE CELL)
# Load models from LoRA paths -> compute VarNLL -> select per label
# -> visualize per-label distributions
# ================================================================

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


# ------------------------------------------------------------
# 2) Compute VarNLL scores from global/client model paths
# ------------------------------------------------------------
@torch.no_grad()
def compute_varnll_scores_from_paths(
    global_path,
    client_paths,
    candidate_loader,
    base_model_name,
    device
):
    """
    Computes per-example VarNLL over probe clients:
        VarNLL(x,y) = Var_i [ NLL_i(x,y) ]

    Also returns labels for per-label selection and optional
    global/client mean NLL statistics for debugging.

    Returns:
        varnll_scores : list[float]
        label_list    : list[label]
        debug_stats   : dict with optional summary stats
    """
    # Load models
    global_model = base_model_loader(global_path ).to(device)
    probe_clients = [base_model_loader(p).to(device) for p in client_paths]

    varnll_scores = []
    label_list = []

    # Optional debugging aggregates
    global_nll_sum = 0.0
    client_mean_nll_sum = 0.0
    total_examples = 0

    for batch in candidate_loader:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        # ------------------------------------------------
        # GLOBAL MODEL NLL (optional debugging only)
        # ------------------------------------------------
        out_g = global_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        logits_g = out_g.logits
        vocab = logits_g.size(-1)

        ce_g = F.cross_entropy(
            logits_g.view(-1, vocab),
            labels.view(-1),
            reduction="none",
            ignore_index=-100
        ).view(labels.size())  # [B, T]

        nll_g = F.cross_entropy(logits_g, labels, reduction="none")

        # ------------------------------------------------
        # CLIENT MODELS NLL
        # ------------------------------------------------
        client_nlls = []
        for client in probe_clients:
            out_i = client(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            logits_i = out_i.logits

            ce_i = F.cross_entropy(
                logits_i.view(-1, vocab),
                labels.view(-1),
                reduction="none",
                ignore_index=-100
            ).view(labels.size())  # [B, T]

            nll_i =  F.cross_entropy(logits_i, labels, reduction="none")
            client_nlls.append(nll_i)

        client_nlls = torch.stack(client_nlls, dim=0)  # [W, B]

        # ------------------------------------------------
        # VarNLL(x,y) = variance over clients
        # ------------------------------------------------
        varnll_batch = torch.var(client_nlls, dim=0, unbiased=False)  # [B]
        varnll_scores.extend(varnll_batch.detach().cpu().tolist())

        batch_labels = batch["labels"]
        if isinstance(batch_labels, torch.Tensor):
            batch_labels = batch_labels.detach().cpu().tolist()
        else:
            batch_labels = list(batch_labels)

        label_list.extend(batch_labels)

        # Optional debugging stats
        client_mean_nll = client_nlls.mean(dim=0)
        global_nll_sum += nll_g.sum().item()
        client_mean_nll_sum += client_mean_nll.sum().item()
        total_examples += nll_g.size(0)

    debug_stats = {
        "global_mean_nll": global_nll_sum / max(total_examples, 1),
        "client_mean_nll": client_mean_nll_sum / max(total_examples, 1),
        "n_examples": total_examples,
        "n_probe_clients": len(client_paths)
    }

    return varnll_scores, label_list, debug_stats

# ------------------------------------------------------------
# 3) Per-label mean + k*std selection for VarNLL
# ------------------------------------------------------------
def select_by_mean_std_per_label(scores, labels, k=1.0, min_keep=1, max_keep=None):
    """
    Select per label using:
        score >= mean(label) + k * std(label)

    Then enforce:
        - at least min_keep examples per label
        - at most max_keep examples per label (if provided)

    Args:
        scores   : list/np.ndarray of VarNLL scores
        labels   : list/np.ndarray of class labels
        k        : threshold multiplier
        min_keep : minimum examples to keep per label
        max_keep : maximum examples to keep per label (None = no upper limit)

    Returns:
        selected_indices : list[int]
        info             : dict[label] = {
                               "mu": float,
                               "sigma": float,
                               "threshold": float,
                               "selected": list[int],
                               "n_total": int,
                               "n_selected": int
                           }
    """
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels)

    if len(scores) != len(labels):
        raise ValueError("scores and labels must have same length")

    if min_keep is not None and min_keep < 0:
        raise ValueError("min_keep must be >= 0")

    if max_keep is not None and max_keep < 0:
        raise ValueError("max_keep must be >= 0")

    if max_keep is not None and min_keep is not None and max_keep < min_keep:
        raise ValueError("max_keep must be >= min_keep")

    selected_indices = []
    info = {}
    groups = defaultdict(list)

    # Group examples by label
    for idx, lab in enumerate(labels):
        if isinstance(lab, (list, tuple, np.ndarray)):
            lab_key = lab[0]
        else:
            lab_key = lab
        groups[lab_key].append(idx)

    for lab, idxs in groups.items():
        idxs = np.asarray(idxs, dtype=int)
        s = scores[idxs]

        if len(s) == 0:
            info[lab] = {
                "mu": 0.0,
                "sigma": 0.0,
                "threshold": 0.0,
                "selected": [],
                "n_total": 0,
                "n_selected": 0
            }
            continue

        mu = float(s.mean())
        sigma = float(s.std())
        threshold = mu + k * sigma

        # Initial threshold selection
        chosen_mask = (s >= threshold)
        chosen = idxs[chosen_mask]

        # Sort all examples in this label by descending score
        order_desc = np.argsort(-s)
        sorted_idxs = idxs[order_desc]

        # Enforce minimum keep
        if min_keep is not None and len(chosen) < min_keep:
            chosen = sorted_idxs[:min(min_keep, len(sorted_idxs))]

        # Enforce maximum keep
        if max_keep is not None and len(chosen) > max_keep:
            chosen_scores = scores[chosen]
            chosen_order = np.argsort(-chosen_scores)
            chosen = chosen[chosen_order[:max_keep]]

        chosen = chosen.tolist()
        selected_indices.extend(chosen)

        info[lab] = {
            "mu": mu,
            "sigma": sigma,
            "threshold": threshold,
            "selected": chosen,
            "n_total": int(len(idxs)),
            "n_selected": int(len(chosen))
        }

    return selected_indices, info

# ------------------------------------------------------------
# 4) Visualization: per-label VarNLL histograms with thresholds
# ------------------------------------------------------------
def visualize_varnll_per_label(scores, labels, info, bins=25, cols=2):
    """
    For each label:
      - plot histogram of VarNLL scores
      - draw vertical line at mean
      - draw vertical line at threshold = mean + k*std
    """
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels)

    unique_labels = list(info.keys())
    n_labels = len(unique_labels)
    rows = int(np.ceil(n_labels / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 4.5 * rows))
    axes = np.array(axes).reshape(-1)

    for ax_idx, lab in enumerate(unique_labels):
        ax = axes[ax_idx]
        idxs = np.where(labels == lab)[0]
        s = scores[idxs]

        mu = info[lab]["mu"]
        sigma = info[lab]["sigma"]
        threshold = info[lab]["threshold"]
        n_total = info[lab]["n_total"]
        n_selected = info[lab]["n_selected"]

        ax.hist(s, bins=bins, alpha=0.75, color="mediumpurple", edgecolor="white")
        ax.axvline(mu, color="orange", linestyle="--", linewidth=2, label=f"mean={mu:.4f}")
        ax.axvline(threshold, color="red", linestyle="-", linewidth=2,
                   label=f"thr={threshold:.4f}")
        x = {1:"positive ", 0:"negative "}
        ax.set_title(f"Label: {x[lab]} , selected={n_selected}")
        ax.set_xlabel("VarNLL")

In [ ]:
def print_selection_summary(info, name="varrel"):
    print("=" * 80)
    print(f"Per-label {name} Selection Summary")
    print("=" * 80)

    total_selected = 0
    total_examples = 0

    for lab, d in info.items():
        mu = d["mu"]
        sigma = d["sigma"]
        thr = d["threshold"]
        n_sel = d["n_selected"]
        n_tot = d["n_total"]

        total_selected += n_sel
        total_examples += n_tot

        print(
            f"Label = {lab:<12} | "
            f"μ = {mu:.6f} | σ = {sigma:.6f} | thr = {thr:.6f} | "
            f"selected = {n_sel}/{n_tot}"
        )

    print("-" * 80)
    print(f"TOTAL selected: {total_selected}  out of  {total_examples}")
    print("=" * 80)

In [ ]:
def select_lowest_by_mean_std_per_label(scores, labels, k=1.0, min_keep=1, max_keep=None):
    """
    Select per label using:
        score <= mean(label) - k * std(label)

    Then enforce:
        - at least min_keep examples per label
        - at most max_keep examples per label, if provided

    Lower scores are preferred.

    Args:
        scores   : list/np.ndarray of VarNLL scores
        labels   : list/np.ndarray of class labels
        k        : threshold multiplier
        min_keep : minimum examples to keep per label
        max_keep : maximum examples to keep per label, None = no upper limit

    Returns:
        selected_indices : list[int]
        info             : dict[label] = {
                               "mu": float,
                               "sigma": float,
                               "threshold": float,
                               "selected": list[int],
                               "n_total": int,
                               "n_selected": int
                           }
    """
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels)

    if len(scores) != len(labels):
        raise ValueError("scores and labels must have same length")

    if min_keep is not None and min_keep < 0:
        raise ValueError("min_keep must be >= 0")

    if max_keep is not None and max_keep < 0:
        raise ValueError("max_keep must be >= 0")

    if max_keep is not None and min_keep is not None and max_keep < min_keep:
        raise ValueError("max_keep must be >= min_keep")

    selected_indices = []
    info = {}
    groups = defaultdict(list)

    # Group examples by label
    for idx, lab in enumerate(labels):
        if isinstance(lab, (list, tuple, np.ndarray)):
            lab_key = lab[0]
        else:
            lab_key = lab
        groups[lab_key].append(idx)

    for lab, idxs in groups.items():
        idxs = np.asarray(idxs, dtype=int)
        s = scores[idxs]

        if len(s) == 0:
            info[lab] = {
                "mu": 0.0,
                "sigma": 0.0,
                "threshold": 0.0,
                "selected": [],
                "n_total": 0,
                "n_selected": 0
            }
            continue

        mu = float(s.mean())
        sigma = float(s.std())
        threshold = mu - k * sigma

        # Initial threshold selection: lowest scores
        chosen_mask = s <= threshold
        chosen = idxs[chosen_mask]

        # Sort all examples in this label by ascending score
        order_asc = np.argsort(s)
        sorted_idxs = idxs[order_asc]

        # Enforce minimum keep
        if min_keep is not None and len(chosen) < min_keep:
            chosen = sorted_idxs[:min(min_keep, len(sorted_idxs))]

        # Enforce maximum keep
        if max_keep is not None and len(chosen) > max_keep:
            chosen_scores = scores[chosen]
            chosen_order = np.argsort(chosen_scores)
            chosen = chosen[chosen_order[:max_keep]]

        chosen = chosen.tolist()
        selected_indices.extend(chosen)

        info[lab] = {
            "mu": mu,
            "sigma": sigma,
            "threshold": threshold,
            "selected": chosen,
            "n_total": int(len(idxs)),
            "n_selected": int(len(chosen))
        }

    return selected_indices, info

In [ ]:
def select_closest_to_mean_per_label(scores, labels, k=1.0, min_keep=1, max_keep=None):
    """
    Select per label using:
        abs(score - mean(label)) <= k * std(label)

    Then enforce:
        - at least min_keep examples per label
        - at most max_keep examples per label, if provided

    Scores closest to the label mean are preferred.

    Args:
        scores   : list/np.ndarray of VarNLL scores
        labels   : list/np.ndarray of class labels
        k        : threshold multiplier
        min_keep : minimum examples to keep per label
        max_keep : maximum examples to keep per label, None = no upper limit

    Returns:
        selected_indices : list[int]
        info             : dict[label] = {
                               "mu": float,
                               "sigma": float,
                               "threshold": float,
                               "selected": list[int],
                               "n_total": int,
                               "n_selected": int
                           }
    """
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels)

    if len(scores) != len(labels):
        raise ValueError("scores and labels must have same length")

    if min_keep is not None and min_keep < 0:
        raise ValueError("min_keep must be >= 0")

    if max_keep is not None and max_keep < 0:
        raise ValueError("max_keep must be >= 0")

    if max_keep is not None and min_keep is not None and max_keep < min_keep:
        raise ValueError("max_keep must be >= min_keep")

    selected_indices = []
    info = {}
    groups = defaultdict(list)

    # Group examples by label
    for idx, lab in enumerate(labels):
        if isinstance(lab, (list, tuple, np.ndarray)):
            lab_key = lab[0]
        else:
            lab_key = lab
        groups[lab_key].append(idx)

    for lab, idxs in groups.items():
        idxs = np.asarray(idxs, dtype=int)
        s = scores[idxs]

        if len(s) == 0:
            info[lab] = {
                "mu": 0.0,
                "sigma": 0.0,
                "threshold": 0.0,
                "selected": [],
                "n_total": 0,
                "n_selected": 0
            }
            continue

        mu = float(s.mean())
        sigma = float(s.std())
        threshold = k * sigma

        distances = np.abs(s - mu)

        # Initial threshold selection: close to mean
        chosen_mask = distances <= threshold
        chosen = idxs[chosen_mask]

        # Sort all examples in this label by distance to mean
        order_closest = np.argsort(distances)
        sorted_idxs = idxs[order_closest]

        # Enforce minimum keep
        if min_keep is not None and len(chosen) < min_keep:
            chosen = sorted_idxs[:min(min_keep, len(sorted_idxs))]

        # Enforce maximum keep
        if max_keep is not None and len(chosen) > max_keep:
            chosen_distances = np.abs(scores[chosen] - mu)
            chosen_order = np.argsort(chosen_distances)
            chosen = chosen[chosen_order[:max_keep]]

        chosen = chosen.tolist()
        selected_indices.extend(chosen)

        info[lab] = {
            "mu": mu,
            "sigma": sigma,
            "threshold": threshold,
            "selected": chosen,
            "n_total": int(len(idxs)),
            "n_selected": int(len(chosen))
        }

    return selected_indices, info

# Ablation

In [ ]:
# Paper metrics: highest sustained accuracy (Eq. 42) and first round sustaining the common accuracy (Eqs. 43-44)
def highest_sustained(acc, W=5, delta=0.02):
    best = None
    for t in range(len(acc) - W):
        if all(acc[t + j] >= acc[t] - delta for j in range(1, W + 1)):
            best = acc[t] if best is None else max(best, acc[t])
    return best

def first_round_sustained(acc, target, W=5, delta=0.02):
    for t in range(len(acc) - W):
        if acc[t] >= target and all(acc[t + j] >= target - delta for j in range(1, W + 1)):
            return t + 1
    return None

In [ ]:
import json, glob

PROBE_MODE = "round0"   # "round0": V(z) from the clients of round 0 (Algorithm 1 as written, cross-silo)
                        # "all":    separate warm-up round with every client (paper's cross-device option);
                        #           S_0 still aggregates only its own models, the extra models are used for V(z) only
PROBE_DIR = "./probe_cache"

S0 = [int(e) for e in get_trained_client(4, num_of_client, 0)]           # clients scheduled for round 0
probe_clients = S0 if PROBE_MODE == "round0" else list(range(num_of_client))
print("S_0 =", S0, "| probe clients:", len(probe_clients))


def reseed():
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def reset_run_state(restore_round0=False):
    """Back to w^0 with no optimizer state; optionally restore the shared round-0 client models + states."""
    for d in ["client_state", "round_cache"]:
        shutil.rmtree(d, ignore_errors=True)
    shutil.rmtree(Main_path, ignore_errors=True)
    shutil.copytree("./pre_model", Main_path)
    if restore_round0:
        shutil.copytree(os.path.join(PROBE_DIR, "round_0"), os.path.join("round_cache", "round_0"))
        shutil.copytree(os.path.join(PROBE_DIR, "client_state"), "client_state")
    reseed()


def make_loader_from_idx(idx):
    """DataLoader over hf_valid[idx] (used for reference sets and for the held-out tuning slice)."""
    sub_ds = hf_valid.select(list(idx))
    tok = sub_ds.map(preprocess_fn, batched=True, remove_columns=sub_ds.column_names)
    tok.set_format(type="torch", columns=cols)
    return DataLoader(tok, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)


@torch.no_grad()
def client_loss_matrix(client_dirs, loader, device):
    """Per-sample NLL of every client model: array [N_clients, N_samples] (same sample order as `loader`)."""
    rows = []
    for d in client_dirs:
        m = base_model_loader(d).to(device).eval()
        net = torch.nn.DataParallel(m) if N_GPU > 1 else m
        out = []
        for batch in loader:
            logits = net(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device)).logits
            out.append(F.cross_entropy(logits, batch["labels"].to(device), reduction="none").float().cpu())
        rows.append(torch.cat(out).numpy())
        del net, m
        torch.cuda.empty_cache()
    return np.stack(rows)

In [ ]:
# ===== Algorithm 1, lines 1-2: round-0 local training once, then V(z) on D_eval =====
prev = [p for p in glob.glob("/kaggle/input/**/probe_cache/done.json", recursive=True)]
if not os.path.exists(os.path.join(PROBE_DIR, "done.json")) and prev:
    shutil.copytree(os.path.dirname(prev[0]), PROBE_DIR)          # resume: reuse a probe stage from an earlier session
    print("copied probe stage from", os.path.dirname(prev[0]))

if os.path.exists(os.path.join(PROBE_DIR, "done.json")):
    meta = json.load(open(os.path.join(PROBE_DIR, "done.json")))
    assert meta["seed"] == seed and meta["S0"] == S0, "probe cache belongs to a different seed / schedule"
    varnll_scores = np.load(os.path.join(PROBE_DIR, "varnll_scores.npy")).tolist()
    labels = np.load(os.path.join(PROBE_DIR, "labels.npy")).tolist()
    _cl = os.path.join(PROBE_DIR, "client_losses.npy")
    client_losses = np.load(_cl) if os.path.exists(_cl) else None
    print("loaded probe stage:", meta)
else:
    reset_run_state()
    torch.cuda.empty_cache(); gc.collect()
    probe_dirs = train_secondorder_fedsophia(
        base_model_loader,
        global_adapter_dir = Main_path,
        train_loaders = train_loader,
        test_loader = test_loader,
        valid_loaders = None,
        algorithm="wagg",
        grad_clip=1.0,
        lr=LR,
        epochs=EPOCHS,
        rounds=rounds,                     # same cosine schedule as the real runs -> round-0 lr is identical
        num_clients=num_of_client,
        clients_per_round=num_of_Part,
        seed=seed,
        device=device,
        round0_clients=probe_clients,
        probe_only=True)

    varnll_scores, labels, debug_stats = compute_varnll_scores_from_paths(
        global_path=Main_path,             # = w^0
        client_paths=probe_dirs,
        candidate_loader=valid_loader,     # D_eval
        base_model_name=MODEL_NAME,
        device=device)
    print("Debug stats:", debug_stats)

    # per-client losses (needed by the K-stability diagnostic); must reproduce V(z) exactly
    client_losses = client_loss_matrix(probe_dirs, valid_loader, device)
    assert client_losses.shape[1] == len(varnll_scores)
    assert np.allclose(client_losses.astype(np.float64).var(axis=0), varnll_scores, rtol=1e-3, atol=1e-5), \
        "client-loss matrix disagrees with V(z)"

    # keep only what the variants need: S_0 models + their optimizer states
    shutil.rmtree(PROBE_DIR, ignore_errors=True)
    for e in S0:
        shutil.copytree(get_round_adapter_dir(0, e), os.path.join(PROBE_DIR, "round_0", f"edge_{e}"))
        os.makedirs(os.path.join(PROBE_DIR, "client_state", f"edge_{e}"), exist_ok=True)
        shutil.copy(get_client_state_paths(e)[1], os.path.join(PROBE_DIR, "client_state", f"edge_{e}", "optimizer.pt"))
    np.save(os.path.join(PROBE_DIR, "varnll_scores.npy"), np.asarray(varnll_scores))
    np.save(os.path.join(PROBE_DIR, "labels.npy"), np.asarray(labels))
    np.save(os.path.join(PROBE_DIR, "client_losses.npy"), client_losses)
    json.dump({"seed": seed, "S0": S0, "probe_mode": PROBE_MODE, "n_probe_clients": len(probe_dirs),
               "max_length": MAX_LENGTH}, open(os.path.join(PROBE_DIR, "done.json"), "w"))
    reset_run_state()                      # free disk: the extra warm-up models are not needed any more
    torch.cuda.empty_cache(); gc.collect()

print("scored samples:", len(varnll_scores))

# Choosing the reference-set size

The paper's rule has **no fixed size**: $K_{bal}=\max_c |\{z: V(z)>\mu_c+\lambda\sigma_c\}|$ (Eq. 10) and every class keeps its top-$K_{bal}$ samples (Eq. 11),
so the size is set by $\lambda$ and by how heavy the tail of $V(z)$ is. Cost per round is $(M+1)\,K$ forward passes (Eq. 37), so $K$ is linear in cost.
* too small $\to$ the average loss on $D^*_{val}$ is noisy and the $1/L$ client weights are noisy;
* too large $\to$ top-$K$ starts to include low-variance samples and the signal is diluted (the limit is the `full` variant).

Two tools below (neither touches the test set):
1. **No training**: $K_{bal}$ for several $\lambda$, and how stable the *ranking of client models* is when the selected set is bootstrap-resampled, per $K$.
2. **Optional training sweep** (`RUN_TRAIN_SWEEP = True`): train once per candidate $K$ and compare on a held-out slice of $D_{eval}$ that is excluded from the selection pool.

The size itself is set by `SIZE_MODE` in the first cell below: `"fixed"` (default, exactly 50 per label = 50 / 50), `"range"` (paper's $K_{bal}$ clipped to `[K_MIN, K_MAX]`) or `"paper"` ($K_{bal}$ as is). Setting `CHOSEN_PER_LABEL` overrides all three. The high / mid / low / random sets all use the same count.

In [ ]:
# ===== how many reference samples? =====
SIZE_MODE        = "fixed"   # "fixed": exactly FIXED_PER_LABEL per label (stable, e.g. 50 / 50)
                             # "range": paper's K_bal (Eq. 10), kept inside [K_MIN, K_MAX] per label
                             # "paper": paper's K_bal as is
FIXED_PER_LABEL  = 50        # used when SIZE_MODE == "fixed"  (50 per label x 2 labels = 100 samples)
K_MIN, K_MAX     = 25, 100   # used when SIZE_MODE == "range"
CHOSEN_PER_LABEL = None      # <- optional: an int (e.g. the suggestion printed below) overrides all of the above
# every mode gives all labels the same count, and high / mid / low / random all use that same count
LAMBDA      = 2.0
K_GRID      = [5, 10, 25, 50, 100, 200, 400]     # candidate samples per label
STAB_TARGET = 0.90                                # 5th-percentile rank correlation we require

scores_arr = np.asarray(varnll_scores, dtype=np.float64)
labels_arr = np.asarray(labels)

# ---- (a) what the paper's rule gives for different lambda (Eq. 10) ----
print("lambda -> per-label counts above mu_c + lambda*sigma_c, and K_bal = max over labels")
for lam in [1.0, 1.5, 2.0, 2.5, 3.0]:
    _, inf = select_by_mean_std_per_label(scores_arr, labels_arr, k=lam, min_keep=0, max_keep=None)
    counts = {int(l): d["n_selected"] for l, d in inf.items()}
    print(f"  lambda={lam:<4}  counts={counts}  K_bal={max(counts.values())}")

# ---- (b) stability of the client ranking vs. K (no training) ----
def _rank(x): return np.argsort(np.argsort(x))
def spearman(a, b): return float(np.corrcoef(_rank(a), _rank(b))[0, 1])

def rank_stability(L, idx, n_boot=300, seed=0):
    """Resample the selected set with replacement; how well does the client ranking (by mean loss) survive?"""
    rng = np.random.default_rng(seed); idx = np.asarray(idx)
    base = L[:, idx].mean(axis=1)
    r = [spearman(base, L[:, rng.choice(idx, size=len(idx), replace=True)].mean(axis=1)) for _ in range(n_boot)]
    return float(np.nanmean(r)), float(np.nanpercentile(r, 5))

if client_losses is None:
    print("\n(no client_losses.npy in the probe cache - delete probe_cache/ and re-run the probe cell to enable the stability check)")
    suggest = None
else:
    pool_min = int(np.bincount(labels_arr).min())
    rows = []
    for kpl in [k for k in K_GRID if k <= pool_min]:
        idx, _ = select_by_mean_std_per_label(scores_arr, labels_arr, k=LAMBDA, min_keep=kpl, max_keep=kpl)
        m, p5 = rank_stability(client_losses, idx)
        rows.append((kpl, len(idx), float(scores_arr[idx].mean()), m, p5))
    print(f"\nclient-ranking stability under bootstrap of the selected set ({client_losses.shape[0]} probe clients)")
    print(f"{'K/label':>8}{'n':>6}{'mean V(z)':>12}{'mean rho':>10}{'p05 rho':>10}")
    for r in rows:
        print(f"{r[0]:>8}{r[1]:>6}{r[2]:>12.5f}{r[3]:>10.3f}{r[4]:>10.3f}")
    ok = [r for r in rows if r[4] >= STAB_TARGET]
    suggest = ok[0][0] if ok else None
    print(f"\nsmallest K with p05 rho >= {STAB_TARGET}: {suggest} per label" if suggest else
          f"\nno K in the grid reaches p05 rho >= {STAB_TARGET}: use a larger K, a smaller lambda, or more probe clients")

    plt.figure(figsize=(6, 3.8))
    plt.plot([r[0] for r in rows], [r[3] for r in rows], "o-", label="mean rho")
    plt.plot([r[0] for r in rows], [r[4] for r in rows], "s--", label="5th percentile")
    plt.axhline(STAB_TARGET, color="gray", ls=":")
    plt.xscale("log"); plt.xlabel("selected samples per label (K)"); plt.ylabel("Spearman rho of client ranking")
    plt.title("Is the reference set big enough?"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ---- (c) optional: train once per candidate K and compare on a held-out slice of D_eval ----
RUN_TRAIN_SWEEP = False      # True = costs about SWEEP_ROUNDS rounds per candidate K
SWEEP_K      = [10, 25, 50, 100, 200]
SWEEP_ROUNDS = 30            # lr schedule is still the 100-round cosine, we just stop early
TUNE_FRAC    = 0.25          # share of D_eval held out for judging (excluded from the selection pool)
PLATEAU_TOL  = 0.01          # a 2,000-sample slice has ~0.8 pt standard error; do not chase smaller differences

if RUN_TRAIN_SWEEP:
    rng = np.random.default_rng(seed)
    tune_idx, pool_idx = [], []
    for lab in np.unique(labels_arr):
        ids = rng.permutation(np.where(labels_arr == lab)[0]); n_t = int(len(ids) * TUNE_FRAC)
        tune_idx += ids[:n_t].tolist(); pool_idx += ids[n_t:].tolist()
    tune_idx, pool_idx = sorted(tune_idx), np.array(sorted(pool_idx))
    tune_loader = make_loader_from_idx(tune_idx)
    pool_min = int(np.bincount(labels_arr[pool_idx]).min())
    sweep_file = f"k_sweep_seed{seed}.json"
    sweep = json.load(open(sweep_file)) if os.path.exists(sweep_file) else {}

    for kpl in [k for k in SWEEP_K if k <= pool_min]:
        if str(kpl) in sweep:
            print(f"[skip] K={kpl}/label already done"); continue
        sel, _ = select_by_mean_std_per_label(scores_arr[pool_idx], labels_arr[pool_idx], k=LAMBDA, min_keep=kpl, max_keep=kpl)
        idx = pool_idx[sel].tolist()
        print(f"\n{'=' * 20} sweep: K={kpl}/label ({len(idx)} samples) {'=' * 20}")
        reset_run_state(restore_round0=True)
        torch.cuda.empty_cache(); gc.collect()
        res = train_secondorder_fedsophia(
            base_model_loader, global_adapter_dir=Main_path, train_loaders=train_loader,
            test_loader=tune_loader,                        # judged on the held-out slice, not on the test set
            valid_loaders=make_loader_from_idx(idx), algorithm="wagg", grad_clip=1.0, lr=LR, epochs=EPOCHS,
            rounds=rounds, num_clients=num_of_client, clients_per_round=num_of_Part, seed=seed, device=device,
            reuse_round0=True, stop_after=SWEEP_ROUNDS)
        sweep[str(kpl)] = {"n": len(idx), "acc": [r["accuracy"] for r in res]}
        json.dump(sweep, open(sweep_file, "w"))

    ks = sorted(sweep, key=int)
    plateau = {k: float(np.mean(sweep[k]["acc"][-5:])) for k in ks}
    best = max(plateau.values())
    print(f"\n{'K/label':>8}{'n':>6}{'sustained %':>13}{'last-5 mean %':>15}")
    for k in ks:
        s_ = highest_sustained(sweep[k]["acc"])
        print(f"{k:>8}{sweep[k]['n']:>6}{('n/a' if s_ is None else f'{100 * s_:.2f}'):>13}{100 * plateau[k]:>15.2f}")
    pick = next(int(k) for k in ks if plateau[k] >= best - PLATEAU_TOL)
    print(f"\nsmallest K within {100 * PLATEAU_TOL:.1f} pt of the best plateau: {pick} per label ({pick * NUM_LABELS} samples)")
    print(f"-> set CHOSEN_PER_LABEL = {pick} in the first tuning cell, then run the selection / ablation cells")
    print("(single seed: look at the curves in k_sweep_seed*.json before trusting a 1-pt difference)")
else:
    print("training sweep skipped (RUN_TRAIN_SWEEP = False)")

In [ ]:
# ===== Algorithm 1, line 2: build the reference sets =====
LAMBDA    = 2.0                     # paper's threshold factor
MID_K     = 1.0                       # "mid": |V - mu_c| <= MID_K * sigma_c, closest to mu_c first

scores_arr = np.asarray(varnll_scores, dtype=np.float64)
labels_arr = np.asarray(labels)
assert len(scores_arr) == len(labels_arr) == len(hf_valid)

# paper rule (Eqs. 9-11): K_bal = max_c |{V(z) > mu_c + lambda*sigma_c}|
_, info_paper = select_by_mean_std_per_label(scores_arr, labels_arr, k=LAMBDA, min_keep=0, max_keep=None)
K_bal = max(d["n_selected"] for d in info_paper.values())
print_selection_summary(info_paper, name=f"VarNLL (lambda={LAMBDA})")

# ---- size of the reference set: the SAME count for every label (class-balanced, e.g. 50 / 50) ----
if CHOSEN_PER_LABEL:                          # a tuned value always wins
    PER_LABEL, how = int(CHOSEN_PER_LABEL), "CHOSEN_PER_LABEL"
elif SIZE_MODE == "fixed":                    # stable: exactly FIXED_PER_LABEL per label
    PER_LABEL, how = int(FIXED_PER_LABEL), "fixed"
elif SIZE_MODE == "range":                    # paper's K_bal, kept inside [K_MIN, K_MAX]
    PER_LABEL, how = int(np.clip(K_bal, K_MIN, K_MAX)), f"K_bal={K_bal} clipped to [{K_MIN}, {K_MAX}]"
elif SIZE_MODE == "paper":                    # paper's K_bal as is (at least 1)
    PER_LABEL, how = max(int(K_bal), 1), f"K_bal={K_bal}"
else:
    raise ValueError(f"SIZE_MODE must be 'fixed', 'range' or 'paper', got {SIZE_MODE!r}")
assert np.bincount(labels_arr).min() >= PER_LABEL, "not enough samples in a class for this size"
K_TOTAL = PER_LABEL * NUM_LABELS
print(f"\nreference-set size: {PER_LABEL} per label x {NUM_LABELS} labels = {K_TOTAL} samples  [{how}]  (paper K_bal = {K_bal})\n")


def select_random_per_label(labels, per_label, seed):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    out = []
    for lab in np.unique(labels):
        idxs = np.where(labels == lab)[0]
        out.extend(rng.choice(idxs, size=per_label, replace=False).tolist())
    return out


high_idx, info_high = select_by_mean_std_per_label(scores_arr, labels_arr, k=LAMBDA, min_keep=PER_LABEL, max_keep=PER_LABEL)
low_idx,  info_low  = select_lowest_by_mean_std_per_label(scores_arr, labels_arr, k=LAMBDA, min_keep=PER_LABEL, max_keep=PER_LABEL)
mid_idx,  info_mid  = select_closest_to_mean_per_label(scores_arr, labels_arr, k=MID_K, min_keep=PER_LABEL, max_keep=PER_LABEL)

variants = {
    "low_variance":  [int(i) for i in low_idx],
    "random":        [int(i) for i in select_random_per_label(labels_arr, PER_LABEL, seed)]
}

print(f"{'variant':<14}{'n':>6}   per-label      mean V(z)    min          max")
for name, idx in variants.items():
    assert len(idx) == len(set(idx))
    if name != "full":
        assert len(idx) == K_TOTAL, (name, len(idx))
    sc = scores_arr[idx]
    print(f"{name:<14}{len(idx):>6}   {str(np.bincount(labels_arr[idx], minlength=NUM_LABELS).tolist()):<13}"
          f"{sc.mean():.6f}   {sc.min():.6f}   {sc.max():.6f}")
m = {n: scores_arr[variants[n]].mean() for n in ["low_variance"]}

json.dump({"seed": seed, "k_total": K_TOTAL, "lambda": LAMBDA, "mid_k": MID_K, "K_bal_paper_rule": int(K_bal),
           "indices": variants}, open("ablation_indices.json", "w"))

In [ ]:
def plot_selected_variants(scores, labels, variants, bins=40):
    scores = np.asarray(scores); labels = np.asarray(labels)
    colors = {"low_variance": "blue", "random": "black"}
    uniq = np.unique(labels)
    fig, axes = plt.subplots(1, len(uniq), figsize=(7 * len(uniq), 4.5))
    axes = np.atleast_1d(axes)
    for ax, lab in zip(axes, uniq):
        ax.hist(scores[labels == lab], bins=bins, alpha=0.6, color="mediumpurple", edgecolor="white")
        ax.set_yscale("log")
        for j, (name, c) in enumerate(colors.items()):
            sel = [i for i in variants[name] if labels[i] == lab]
            ax.plot(scores[sel], np.full(len(sel), 0.6 * 1.7 ** j), "|", color=c, markersize=14, label=name)
        ax.set_title(f"label {lab}: V(z) over D_eval and the selected samples")
        ax.set_xlabel("V(z)"); ax.set_ylabel("count (log)")
        ax.legend()
    plt.tight_layout(); plt.show()

plot_selected_variants(scores_arr, labels_arr, variants)

In [ ]:
def make_wagg_loader(name):
    """Reference-set loader for a variant; 'full' = all of D_eval."""
    if name == "full":
        return valid_loader
    return make_loader_from_idx(variants[name])

In [ ]:
# ===== Algorithm 1, lines 3-11 for every variant =====
STRATEGIES_TO_RUN = ["low_variance", "random"]   # finished ones are skipped

for name in STRATEGIES_TO_RUN:
    out_file = f"ablation_results_{name}.json"
    if os.path.exists(out_file):
        print(f"[skip] {name}: {out_file} exists")
        continue
    print(f"\n{'=' * 30} VARIANT: {name}  (|D*_val| = {len(variants[name])}) {'=' * 30}")
    reset_run_state(restore_round0=True)          # w^0 + the shared round-0 client models
    torch.cuda.empty_cache(); gc.collect()

    start_time = time.time()
    results = train_secondorder_fedsophia(
        base_model_loader,
        global_adapter_dir = Main_path,
        train_loaders = train_loader,
        test_loader = test_loader,
        valid_loaders = make_wagg_loader(name),
        algorithm="wagg",
        grad_clip=1.0,
        lr=LR,
        epochs=EPOCHS,
        rounds=rounds,
        num_clients=num_of_client,
        clients_per_round=num_of_Part,
        seed=seed,
        device=device,
        reuse_round0=True)
    elapsed = time.time() - start_time

    json.dump({"strategy": name, "seed": seed, "n_reference": len(variants[name]),
               "elapsed_sec": elapsed, "results": results}, open(out_file, "w"))
    print(f"[done] {name}: {elapsed / 3600:.2f} h -> {out_file}")

# Results

In [ ]:
curves = {}
for name in [ "low_variance", "random"]:
    p = f"ablation_results_{name}.json"
    if os.path.exists(p):
        curves[name] = [r["accuracy"] for r in json.load(open(p))["results"]]

sus = {n: highest_sustained(a) for n, a in curves.items()}
ok = [v for v in sus.values() if v is not None]
a_com = min(ok) if ok else None
print("A_com =", None if a_com is None else f"{100 * a_com:.3f} %")
print(f"{'variant':<15}{'highest sustained %':>20}{'round to A_com':>16}{'final %':>10}")
for n, a in curves.items():
    s_ = "n/a" if sus[n] is None else f"{100 * sus[n]:.2f}"
    r_ = first_round_sustained(a, a_com) if a_com is not None else None
    print(f"{n:<15}{s_:>20}{str(r_):>16}{100 * a[-1]:>10.2f}")

plt.figure(figsize=(8, 4.5))
for n, a in curves.items():
    plt.plot(range(1, len(a) + 1), [100 * x for x in a], label=n)
plt.xlabel("communication round"); plt.ylabel("test accuracy (%)")
plt.title(f"Reference-set selection ablation (Sentiment140 non-IID, seed {seed})")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"ablation_seed{seed}.png", dpi=200); plt.show()